# 🌍 AfriSpeech Selector — build a training set in the cloud

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfriSpeech/afrispeech-selector/blob/main/notebooks/afrispeech_selector.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/AfriSpeech/afrispeech-selector/blob/main/notebooks/afrispeech_selector.ipynb)

Select African languages from [`AfriSpeech/african-speech-public_v1`](https://huggingface.co/datasets/AfriSpeech/african-speech-public_v1) by recorded **hours**, pull a sized sample, and download it or push it to your own HF dataset repo — in a training-ready schema (`audio · text · language · country · length`).

### Runtime: CPU is fine (no GPU needed)
Building a dataset is download/IO, not compute — the **default CPU runtime works**. You only need internet access:
- **Kaggle:** turn **Internet ON** (`Settings → Internet`).
- **Colab:** the default CPU runtime already has internet.

(If you go on to fine-tune in the same session, switch to a GPU then.)

In [ ]:
# No GPU required — quick environment check
import platform, os
print('Python', platform.python_version(), '| CPUs:', os.cpu_count())

## 1. Install

In [ ]:
# Installs the `afrispeech-select` CLI + library and the audio backend.
%pip install -q "afrispeech-selector @ git+https://github.com/AfriSpeech/afrispeech-selector.git" torchcodec
print('Installed. If a torchcodec wheel is unavailable for this runtime, the streamed mp3 bytes still download fine.')

## 2. Browse the languages
Every subset, ranked by hours.

In [ ]:
!afrispeech-select --list-langs | head -40

## 3. Preview a selection (no download)
Top 10 languages, country-balanced (max 2 per country), 200 clips each. Tweak the flags, then re-run.

In [ ]:
!afrispeech-select --top 10 --max-per-country 2 --per-language 200 --dry-run

## 4. Build the dataset
Streams only the samples you ask for. Try other knobs:
- `--max-hours-per-lang 0.5` — duration budget per language (decimals OK)
- `--min-clip-sec 3 --max-clip-sec 20` — keep only clips in this length window
- `--languages twi_twi,hausa_hau` — hand-pick instead of `--top`
- `--split train|val|test|all`

In [ ]:
!afrispeech-select --top 10 --max-per-country 2 --per-language 200 \
    --out afrispeech_data --format disk,parquet

## 5. Inspect & listen

In [ ]:
from datasets import load_from_disk
from IPython.display import Audio, display
import collections

ds = load_from_disk('afrispeech_data')
print(ds)
print('clips per language:', dict(collections.Counter(ds['language'])))

ex = ds[0]
print(f"\n{ex['language']} ({ex['country']}) — {ex['length']:.1f}s\n{ex['text'][:200]}")
a = ex['audio']
try:
    arr, sr = a['array'], a['sampling_rate']            # classic dict form
except (TypeError, KeyError):
    s = a.get_all_samples(); arr, sr = s.data.numpy().squeeze(), s.sample_rate  # torchcodec
display(Audio(arr, rate=sr))

## 6. (Optional) Push to your own HF dataset repo
Needs a token with **write** access from https://huggingface.co/settings/tokens.

In [ ]:
import os, getpass
os.environ['HF_TOKEN'] = getpass.getpass('HF write token: ')
REPO = 'your-username/my-afrispeech-subset'  # <-- change me

!afrispeech-select --top 10 --max-per-country 2 --per-language 200 \
    --out afrispeech_data --push {REPO} --token $HF_TOKEN

## 7. Use it for training
The output is a standard 🤗 `datasets` dataset — feed it straight to a trainer:

```python
from datasets import load_from_disk
ds = load_from_disk('afrispeech_data').train_test_split(test_size=0.1)
# ds['train'] / ds['test'] with columns: audio, text, language, country, length, iso, subset
```

Run `!afrispeech-select --help` for every option.